# 3. 线性神经网络（上）

> 线性回归相关

## 3.1 线性回归的原理与矢量化

### 1. 模型公式：矢量的表达 (Vectorization)
在数学书里，线性方程可能写成 $y = w_1x_1 + w_2x_2 + b$。但在深度学习里，我们必须用**矩阵**来思考。

*   **数学公式**：$\mathbf{\hat{y}} = \mathbf{X}\mathbf{w} + b$
*   **维度拆解**（假设有 1000 个样本，2 个特征）：
    *   $\mathbf{X}$ (特征矩阵)：形状为 `(1000, 2)`。
    *   $\mathbf{w}$ (权重向量)：形状为 `(2, 1)`。
    *   $b$ (偏置)：是一个标量（通过广播机制加到每个结果上）。
    *   $\mathbf{\hat{y}}$ (预测值)：形状为 `(1000, 1)`。
*   **核心直觉**：矩阵乘法 $\mathbf{X}\mathbf{w}$ 一次性完成了所有样本的加权求和。

---

### 2. 损失函数：均方误差 (MSE)
> 损失函数的选取取决于对数据的原始假设。**MSE** 是基于正态进行的。

我们需要一个数字来衡量“预测得有多离谱”。

*   **公式**：$L(\mathbf{w}, b) = \frac{1}{n} \sum_{i=1}^{n} \frac{1}{2} (\hat{y}^{(i)} - y^{(i)})^2$
*   **为什么要平方？**
    1.  **正值化**：误差有正有负，平方能确保误差永远是正数。
    2.  **惩罚大误差**：平方会放大较大的差距。如果差 2，误差是 4；如果差 10，误差是 100。这迫使模型优先修复那些“离谱”的预测。
*   **那个 $\frac{1}{2}$ 是干嘛的？**
    *   它是为了**求导方便**。当你对 $(\dots)^2$ 求导时，会掉下来一个 $2$，正好和 $\frac{1}{2}$ 约掉。

---

### 3. 优化算法：小批量随机梯度下降 (Minibatch SGD)
这是让模型“学会”东西的魔法。

*   **梯度 (Gradient)**：损失函数对 $w$ 的导数。它告诉我们 $w$ 往哪个方向走，误差增加最快。
*   **反方向更新**：我们要减小误差，所以往梯度的**反方向**走。
*   **数学更新公式**：
    $$\mathbf{w} \leftarrow \mathbf{w} - \eta \cdot \frac{\partial L}{\partial \mathbf{w}}$$
    （其中 $\eta$ 是**学习率**，代表步长）。
*   **为什么要“小批量 (Minibatch)”？**
    *   **全量梯度下降**：算一遍要用全人类的数据，太慢，内存装不下。
    *   **随机梯度下降 (SGD)**：一次只看一个样本，太不稳定，路线太乱。
    *   **小批量**：一次看 10~128 个样本。它是速度和精确度的完美折中，且利用了 GPU 的并行能力。

---

### 4. 正态分布与线性回归（进阶夯实）
为什么我们要用“均方误差”？这里有一个深层数学联系。

*   **理论**：如果假设观测误差符合**正态分布（高斯分布）**，那么通过“最大似然估计”推导出来的最优解，正好就是“均方误差”最小的点。
*   **直觉**：当你写 `labels += torch.normal(0, 0.01, ...)` 时，你就是在模拟这个数学假设。

---

### 总结：本章数学知识的逻辑闭环

1.  **模型** (Linear) 定义了**怎么猜**。
2.  **损失** (MSE) 定义了**猜得有多错**。
3.  **梯度** (Calculus) 找到了**往哪改**。
4.  **优化** (SGD) 执行了**改多少**。


## 3.2 从零开始的线性回归

In [14]:
# 模型公式 与 损失函数
import torch

# 假设有 2 个样本和 3 个特征
X = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
w = torch.tensor([[0.1], [0.2], [0.3]])
b = 0.5

# 1. 模型预测 (Xw + b)
y_hat = torch.matmul(X, w) + b
print("预测值:\n", y_hat)

# 2. 假设真实值是
y_true = torch.tensor([[1.5], [4.0]])

# 3. 计算均方误差损失
loss = (y_hat - y_true)**2 / 2
print("每个样本的损失:\n", loss)
print("平均损失:", loss.mean())

预测值:
 tensor([[1.9000],
        [3.7000]])
每个样本的损失:
 tensor([[0.0800],
        [0.0450]])
平均损失: tensor(0.0625)


In [15]:
# 矢量化计算
import torch
import time

# 创建两个巨大的向量 (1万维)
n = 10000
a = torch.ones(n)
b = torch.ones(n)

# A. 使用 Python 的 for 循环相加
start_time = time.time()
c = torch.zeros(n)
for i in range(n):
    c[i] = a[i] + b[i]
print(f"For 循环耗时: {time.time() - start_time: .5f} 秒")

# B. 使用矢量计算 (直接相加)
start_time = time.time()
d = a + b
print(f"矢量化计算耗时: {time.time() - start_time: .5f} 秒")

For 循环耗时:  0.03650 秒
矢量化计算耗时:  0.00021 秒


In [16]:
# 数据生成器
def synthetic_data(w, b, num_examples):
    """生成 y = Xw + b + 噪声"""
    # 1. 随机生成 X (特征矩阵)
    # 假设有 num_examples 个样本，每个样本特征数等于 len(w)
    X = torch.normal(0, 1, (num_examples, len(w)))

    # 2. 根据公式计算真实的 y
    y = torch.matmul(X, w) + b

    # 3. 给 y 加上一定的噪声
    y += torch.normal(0, 0.01, y.shape)

    return X, y.reshape(-1, 1)

# 设定真实的参数
true_w = torch.tensor([2., -3.4])
true_b = 4.2

# 生成 1000 个数据点
features, labels = synthetic_data(true_w, true_b, 1000)

In [17]:
# 数据读取器
import random
def data_iter(batch_size, features, labels):
    '''
    batch_size: 批处理的数据规模;
    features: 采样集（原始数据）;
    labels: 真实的统计量。
    '''
    num_examples = len(features)
    indices = list(range(num_examples)) # 序列号 [0, 1, ...]
    
    # 把序号打乱，比如变成 [5, 122, 9, ...]
    random.shuffle(indices) 
    
    # 按步长跳跃的循环
    # range(start, stop, step) 
    # 这里是：从 0 开始，到 1000 结束，每次跳 batch_size(10) 个数
    # i 的取值会是：0, 10, 20, 30 ...
    for i in range(0, num_examples, batch_size):
        
        # 切片取序号
        # 这一步是算出当前这一批次的“索引号”是哪些
        # min() 是为了防止最后剩下的元素不够 batch_size 时溢出
        # 比如 i=0 时，取 indices[0: 10]
        batch_indices = torch.tensor(
            indices[i: min(i + batch_size, num_examples)])
        
        # 生成器语法：yield
        # 它是这一段最难理解的地方。
        # 它的作用是：返回这一批次的数据，然后函数“暂停”在这里。
        # 等到下一次循环（下一次要牌）时，再从这里继续跑。
        yield features[batch_indices], labels[batch_indices]

# 测试读取 10 个试试
batch_size = 10
for X, y in data_iter(batch_size, features, labels):
    print(X, '\n', y)
    break # 因为只看第一批，所以 break


tensor([[ 0.4068,  0.7997],
        [ 2.3448,  1.1474],
        [ 0.6897, -0.4112],
        [-1.1868,  0.9856],
        [-1.1071, -1.1169],
        [-0.7069, -0.5189],
        [-0.6388,  0.4129],
        [ 0.2715,  1.7207],
        [ 1.9380,  0.1182],
        [ 1.0961, -1.0586]]) 
 tensor([[ 2.2957],
        [ 5.0085],
        [ 6.9666],
        [-1.5373],
        [ 5.7952],
        [ 4.5140],
        [ 1.5287],
        [-1.1076],
        [ 7.6720],
        [ 9.9973]])


In [18]:
# 初始化模型参数
w = torch.normal(0, 0.01, size=(2, 1), requires_grad=True)
b = torch.zeros(1, requires_grad=True)

In [19]:
# 定义模型和损失函数
def linreg(X, w, b):
    """线性回归模型"""
    return torch.matmul(X, w) + b

def squared_loss(y_hat, y):
    """均方损失"""
    return (y_hat - y.reshape(y_hat.shape))**2 / 2

In [20]:
# 定义优化算法 (SGD: 随机梯度下降)
# 注意：需使用 with torch.no_grad(): 来确保更新参数的过程不产生梯度。
def sgd(params, lr, batch_size):
    """
    小批量随机梯度下降
    
    params: 参数;
    lr: 学习率（步长）;
    batch_size: 批数据规模。
    """
    with torch.no_grad():
        for param in params:
            # 为什么要除以 batch_size？因为我们之前算 loss 的时候没取平均，而是一个 batch 的总和
            param -= lr * param.grad / batch_size
            # 梯度清 0，为下一次计算做准备
            param.grad.zero_()

In [21]:
# 训练循环
lr = 0.03 # 学习率
num_epochs = 3 # 训练 3 轮
net = linreg # 我们定义的模型
loss = squared_loss # 我们定义的损失函数

for epoch in range(num_epochs):
    for X, y in data_iter(batch_size, features, labels):
        l = loss(net(X, w, b), y) # 1. 计算当前预测值的损失
        # 当前 l 的形状是 (batch_size, 1) 而不是标量。用 l.sum() 转换成标量
        l.sum().backward() # 2. 靠反向传播计算梯度
        sgd([w, b], lr, batch_size) # 3. 使用 sgd 更新参数

    # 每一轮练完，看看效果
    with torch.no_grad():
        train_l = loss(net(features, w, b), labels)
        print(f"epoch {epoch + 1}, loss {float(train_l.mean()): f}")

print(f"w 的估计误差: {true_w - w.reshape(true_w.shape)}")
print(f"b 的估计误差: {true_b - b}")

epoch 1, loss  0.052801
epoch 2, loss  0.000239
epoch 3, loss  0.000051
w 的估计误差: tensor([ 0.0002, -0.0013], grad_fn=<SubBackward0>)
b 的估计误差: tensor([0.0009], grad_fn=<RsubBackward1>)


## 3.3 线性回归的简洁实现

> - 产生数据 (Tensor)
>
> - 封装数据 (Dataset)
>
> - 分发数据 (DataLoader)

> 1. torch.utils.data：数据处理模块。
>
> 2. torch.nn：神经网络抽象模块（NN 代表 Neural Network）。
>
> 3. torch.optim：优化算法模块。

In [22]:
# 1. 生成数据集
import torch
from torch import Tensor
from torch.utils import data

def synthetic_data(w: Tensor, b: float, num_examples: int) -> tuple[Tensor, Tensor]:
    """生成 y = Xw + b + 噪声
    
    Args:
        w: 真实的权重向量，形状为 (num_inputs,)。 # num_inputs: 特征的个数
        b: 真实的偏置值。
        num_examples: 生成的样本数量。

    Returns:
        包含特征矩阵和标签向量的元组。
        - features: 形状为 (num_examples, len(w))。
        - labels: 形状为 (num_examples, 1)。
    """
    X: Tensor = torch.normal(0, 1, (num_examples, len(w)))
    y: Tensor = torch.matmul(X, w) + b
    y += torch.normal(0, 0.01, y.shape)
    return X, y.reshape(-1, 1)

true_w: Tensor = torch.tensor([2, -3.4])
true_b: float = 4.2
features, labels = synthetic_data(true_w, true_b, 1000)


In [23]:
# 2. 读取数据集
def load_array(data_arrays: tuple[Tensor, Tensor], batch_size: int, is_train: bool = True) -> data.DataLoader:
    """构造一个 PyTorch 数据迭代器。
    
    Args:
        data_arrays: 包含 (features, labels) 的元组。
        batch_size: 每个批次的样本数。
        is_train: 是否需要在每个 epoch 重新打乱数据。

    Returns:
        PyTorch 的 DataLoader 对象。
    """
    dataset = data.TensorDataset(*data_arrays) # 用 TensorDataset 封装原始数据
    return data.DataLoader(dataset, batch_size, shuffle=is_train)

batch_size: int = 10
data_iter = load_array((features, labels), batch_size)

In [24]:
# 3. 定义模型与初始化参数
from torch import nn

# nn.Linear(输入特征数, 输出特征数)
net = nn.Sequential(nn.Linear(2, 1))

# 初始化参数
# net[0] 代表访问 Sequential 容器里的第一个层
# .data: 直接访问原始值，而不被梯度追踪。
# 现代做法一般是显式写出 with torch.no_grad(): 或用 nn.init
net[0].weight.data.normal_(0, 0.01)
net[0].bias.data.fill_(0)

tensor([0.])

In [25]:
# 4. 定义损失函数与优化算法
# 均方误差损失 (MSE)
loss_fn = nn.MSELoss()
# loss_fn = nn.L1Loss()

# 优化器：随机梯度下降(SGD)
# net.parameters() 会自动提取 w 和 b
trainer = torch.optim.SGD(net.parameters(), lr=0.03)

In [26]:
# 5. 训练循环
def train_liner_model(
        net: nn.Module,
        data_iter: data.DataLoader,
        loss_fn: nn.modules.loss._Loss,
        trainer: torch.optim.Optimizer,
        num_epochs: int
) -> None:
    """训练线性回归模型的核心循环。
    
    Args:
        net: 神经网络模型。
        data_iter: 训练数据迭代器。
        loss_fn: 损失函数。
        trainer: 优化器。
        num_epochs: 训练轮数。
    """
    for epoch in range(num_epochs):
        for X, y in data_iter:
            # 1. 前向传播计算 loss
            l: Tensor = loss_fn(net(X), y)

            # 2. 梯度清零 (清除上一次的记忆)
            trainer.zero_grad()

            # 3. 反向传播
            l.backward()

            # 4. 更新参数
            trainer.step()

        # 打印进度
        with torch.no_grad():
            current_l: Tensor = loss_fn(net(features), labels)
            print(f"epoch {epoch + 1}, loss {current_l: f}")

# 执行训练
train_liner_model(net, data_iter, loss_fn, trainer, num_epochs=3)

# 查看训练结果
print(f"w 的估计误差: {true_w - net[0].weight.data}")
print(f"b 的估计误差: {true_b - net[0].bias.data}")

epoch 1, loss  0.000517
epoch 2, loss  0.000098
epoch 3, loss  0.000098
w 的估计误差: tensor([[-0.0004,  0.0005]])
b 的估计误差: tensor([0.0010])


## 附录：PyTorch 的核心数据类型

### 1. 基础数据类型 (Base Types)

#### `torch.Tensor` (张量)
*   **含义**：PyTorch 的基本计算单位。
*   **本质**：它可以是一个标量、向量、矩阵或多维数组。
*   **特权**：与普通 NumPy 数组不同，它携带了 **梯度（Grad）** 信息，并且可以被放置在 **GPU** 上加速运算。
*   **代码习惯**：通常我们使用 `from torch import Tensor` 来进行类型标注。

#### `typing.Tuple` (元组)
*   **含义**：Python 标准库类型。
*   **用途**：在 PyTorch 中经常用于标注返回多个张量的函数，例如 `Tuple[Tensor, Tensor]` 表示返回一个包含两个张量的元组（通常是特征和标签）。

---

### 2. 数据处理相关 (Data Pipeline Types)

#### `torch.utils.data.TensorDataset` (张量数据集)
*   **含义**：数据的“包装盒”。
*   **用途**：它把特征张量（Features）和标签张量（Labels）一一对应地打包。
*   **直觉**：你可以把它想象成一个两列的 Excel 表格。

#### `torch.utils.data.DataLoader` (数据加载器)
*   **含义**：数据的“分发管理器”。
*   **用途**：它负责从 `TensorDataset` 中抓取数据。它处理 **打乱（Shuffle）** 、 **分批（Batching）** 以及多线程并行加载。
*   **类型标注**：`data.DataLoader`。

#### 核心对比表

| 特性 | Dataset (仓库) | DataLoader (流水线) |
| :--- | :--- | :--- |
| **关注点** | **单个样本** 的存储和读取 | **成批样本** 的组织和分发 |
| **主要参数** | 数据路径、预处理 (Transform) | `batch_size`, `shuffle`, `num_workers` |
| **访问方式** | 通过索引访问：`ds[i]` | 通过循环访问：`for batch in dl` |
| **内存占用** | 通常很小（只存路径或索引） | 取决于 `batch_size` 和缓冲区 |
| **层级关系** | **底层**，被 DataLoader 包裹 | **高层**，包裹着 Dataset |

---

### 3. 模型与架构相关 (NN Types)

#### `torch.nn.Module` (神经网络基类)
*   **含义**：PyTorch 中所有神经网络的 **“祖先”** 。
*   **用途**：无论是简单的线性回归，还是复杂的 GPT，本质上都是一个 `Module`。
*   **核心功能**：它会自动追踪该模型内所有的参数（Weights 和 Biases）。

#### `torch.nn.Sequential` (顺序容器)
*   **含义**：一种特殊的 `Module`，内部像**积木**一样按顺序存放其他层。
*   **用途**：当你的网络是简单的“输入 -> A层 -> B层 -> 输出”时，用它可以极大地简化代码。

#### `torch.nn.Linear` (线性层/全连接层)
*   **含义**：执行 $y = Xw + b$ 计算的封装类。
*   **参数**：它内部自动创建了 `weight` 和 `bias` 张量。

---

### 4. 训练与优化相关 (Training Types)

#### `torch.optim.Optimizer` (优化器基类)
*   **含义**：负责“更新参数”的逻辑抽象。
*   **具体实现**：如 `torch.optim.SGD`（随机梯度下降）或 `torch.optim.Adam`。
*   **工作内容**：它保存了指向模型参数的指针，并根据参数的 `.grad` 属性来执行减法更新。

#### `torch.nn.modules.loss._Loss` (损失函数基类)
*   **含义**：所有损失函数（如 `MSELoss`）的父类。
*   **用途**：用于衡量预测值与真实值之间的差距。
*   **标注建议**：在强类型标注中，虽然可以使用这个基类，但初学者有时也直接标注为 `nn.Module`，因为损失函数在 PyTorch 中本质上也是一个 `Module`。

---

### 5. 补充：一个 PyTorch 对象的内部解剖

当你看到一个 `Tensor` 时，你应该意识到它内部至少包含三个重要的属性：
1.  **`.data`**: 存储的数值本身（如 4.2）。
2.  **`.grad`**: 该数值对应的梯度（如 0.01）。
3.  **`.device`**: 存储的位置（如 `cpu` 或 `cuda:0`）。
